# PTCG Merged Agent Workbench

Parent notebook for **The Pokémon Company - PTCG AI Battle Challenge Simulation**.

## Document map

| Doc | Notebook | Role |
|-----|----------|------|
| **4/8 — Dragapult** | `a-sample-rule-based-agent-dragapult-ex-deck.ipynb` | Base policy skeleton |
| **9/11 — Meta snapshot** | `pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb` | Deck choice + holdout mindset |
| **10 — Expectimax** | `improved-probabilistic-agent.ipynb` | Search API + UCB1 + opponent reads |

## Path conventions

| | **Reads (input)** | **Writes (output)** |
|---|-------------------|---------------------|
| **Local** (`.venv`) | `data/` (e.g. `data/deck.csv`, `data/cg/`) | repo root + `notebooks/` |
| **Kaggle** | `/kaggle/input/` (attached datasets + competition files) | `/kaggle/working/` |

Section 1 resolves paths automatically via `env_paths.py`.

## Concrete merge plan

1. **Doc 4/8:** Dragapult scoring skeleton -> `DragapultPolicy`
2. **Docs 9/11:** meta-informed deck in `data/deck.csv` (local) or attached input dataset (Kaggle)
3. **Doc 10:** Search API + UCB1 on top of `DragapultPolicy` (not Lucario `AdvancedPolicy`)


## 1. Environment + path setup

- **Local:** run with the project `.venv` kernel; reads from `data/`.
- **Kaggle:** reads from `/kaggle/input/` (read-only); all generated files go to `/kaggle/working/`.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

for candidate in (Path.cwd() / "notebooks", Path.cwd()):
    if (candidate / "env_paths.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from env_paths import describe_paths, discover_notebooks_dir, get_paths, stage_deck_for_build

NOTEBOOKS_DIR = discover_notebooks_dir()
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

PATHS = get_paths()
PATHS.ensure_dirs()

print(describe_paths(PATHS))
if PATHS.environment == "local" and not PATHS.using_local_venv:
    print("Tip: select the project .venv kernel for local runs.")

SOURCES = {
    "dragapult (Doc 4/8)": PATHS.ref_dir / "a-sample-rule-based-agent-dragapult-ex-deck.ipynb",
    "expectimax (Doc 10)": PATHS.ref_dir / "improved-probabilistic-agent.ipynb",
    "meta snapshot (Doc 9/11)": PATHS.ref_dir / "pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb",
}

print("\nReference notebooks:")
for label, path in SOURCES.items():
    print(f"  {'OK' if path.exists() else 'MISSING'} - {label}: {path}")


## 2. Meta-informed deck choice (Docs 9/11)

Choose **Starmie** or **Festival Thwackey** for the meta-informed deck list. Agent code in the meta notebooks is base64-only (`main_b64`, `deck_b64`); this workbench uses their **analysis mindset** only.


In [ ]:
META_FIELD = pd.DataFrame([
    {"archetype": "starmie", "usage_pct": 13.85, "score_pct": 51.89, "role": "Primary meta candidate"},
    {"archetype": "festival_thwackey", "usage_pct": 1.16, "score_pct": 45.99, "role": "Low-share dark horse"},
    {"archetype": "dragapult", "usage_pct": 7.31, "score_pct": 49.13, "role": "Policy skeleton constants in builder"},
])

META_FIELD[META_FIELD["archetype"].isin(["starmie", "festival_thwackey"])].sort_values("usage_pct")


In [ ]:
if PATHS.deck_path and PATHS.deck_path.exists():
    deck = [int(line) for line in PATHS.deck_path.read_text().splitlines() if line.strip()]
    assert len(deck) == 60, f"Expected 60 cards, got {len(deck)}"
    print(f"Deck source ({PATHS.environment}): {PATHS.deck_path}")
    print(f"Cards: {len(deck)}, unique ids: {len(set(deck))}")
else:
    if PATHS.environment == "kaggle":
        print("Attach a dataset with deck.csv under /kaggle/input, then re-run.")
    else:
        print("Add data/deck.csv locally, then re-run.")


## 3. Source contributions

- **Doc 4/8:** `DragapultPolicy` — logs, deck reconstruction, combo planning, scoring
- **Doc 10:** `_opponent_is_water_deck`, `_opponent_is_crustle_wall`, `SEARCH_ALGO` + UCB1 (wired to Dragapult, not Lucario `AdvancedPolicy`)
- **Docs 9/11:** deck framing + holdout gates (sections 2 and 5)


## 4. Build merged `main.py`

Writes to `PATHS.main_py` (`/kaggle/working/main.py` on Kaggle, repo-root locally).


In [ ]:
import subprocess

builder = PATHS.notebooks_dir / "build_merged_agent.py"
if not builder.exists():
    builder = NOTEBOOKS_DIR / "build_merged_agent.py"

subprocess.run([sys.executable, str(builder)], check=True, cwd=str(builder.parent))

merged_path = PATHS.merged_main_py
if not merged_path.exists():
    merged_path = builder.parent / "merged_agent_main.py"

main_src = merged_path.read_text(encoding="utf-8")
PATHS.main_py.write_text(main_src, encoding="utf-8")
print(f"Wrote {PATHS.main_py} ({len(main_src.splitlines())} lines)")


In [ ]:
REQUIRED_MARKERS = {
    "DragapultPolicy": "Doc 4/8 policy skeleton",
    "_opponent_is_water_deck": "Doc 10 opponent read",
    "_opponent_is_crustle_wall": "Doc 10 opponent read",
    "SEARCH_ALGO": "Doc 10 search wrapper",
    "Phase 2: UCB1": "Doc 10 UCB1 loop",
    "search_begin": "Doc 10 Search API",
}

main_text = PATHS.main_py.read_text(encoding="utf-8")
checks = pd.DataFrame([
    {"marker": k, "purpose": v, "present": k in main_text}
    for k, v in REQUIRED_MARKERS.items()
])
missing = checks[~checks["present"]]["marker"].tolist()
print(checks.to_string(index=False))
if missing:
    raise RuntimeError(f"Build verification failed - missing: {missing}")
print("Build verification passed.")


## 5. Holdout validation (meta gates)


In [ ]:
HOLDOUT_OPPONENTS = ["starmie", "festival_thwackey", "hop_trevenant", "archaludon", "lucario"]
HOLDOUT_GAMES = 40
PROMOTE_THRESHOLD = 0.52


def run_holdout_suite(opponents=HOLDOUT_OPPONENTS, games=HOLDOUT_GAMES):
    raise NotImplementedError("Wire to cabt / kaggle-environments using PATHS.cg_dir")


def summarize_holdout(results):
    rows = []
    for row in results:
        total = row["wins"] + row["losses"] + row["ties"]
        rate = row["wins"] / total if total else 0.0
        passed = rate >= PROMOTE_THRESHOLD
        rows.append({
            **row,
            "win_rate": rate,
            "holdout_gate": "holdout_pass" if passed else "holdout_fail",
            "verdict": "PROMOTE_CANDIDATE" if passed else "HOLD_DO_NOT_SUBMIT",
        })
    return pd.DataFrame(rows)

print("Stress pool:", HOLDOUT_OPPONENTS)


## 6. Package submission

Output: `PATHS.submission_tar` (`/kaggle/working/submission.tar.gz` on Kaggle).

Reads `deck.csv` and `cg/` from input paths; writes tarball to working/output directory.


In [ ]:
import shutil
import tarfile


def build_submission(output: Path | None = None):
    output = output or PATHS.submission_tar
    deck_file = stage_deck_for_build(PATHS)

    if not PATHS.main_py.exists():
        raise FileNotFoundError(f"Run section 4 first. Missing: {PATHS.main_py}")

    cg_src = PATHS.cg_dir
    if cg_src is None or not (cg_src / "api.py").exists():
        raise FileNotFoundError(
            "cg SDK not found. Local: data/cg/. Kaggle: add competition sample_submission input."
        )

    staging = PATHS.output_root / ".submission_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)

    shutil.copy2(PATHS.main_py, staging / "main.py")
    shutil.copy2(deck_file, staging / "deck.csv")
    shutil.copytree(cg_src, staging / "cg")

    with tarfile.open(output, "w:gz") as tar:
        tar.add(staging / "main.py", arcname="main.py")
        tar.add(staging / "deck.csv", arcname="deck.csv")
        for item in sorted((staging / "cg").rglob("*")):
            if item.is_file() and "__pycache__" not in item.parts:
                tar.add(item, arcname=str(Path("cg") / item.relative_to(staging / "cg")))

    shutil.rmtree(staging)
    print(f"Created {output} ({output.stat().st_size / 1024 / 1024:.2f} MiB)")


# build_submission()


## 7. Checklist

1. Section 1 - verify environment + paths
2. Section 2 - deck.csv in `data/` (local) or `/kaggle/input/` (Kaggle)
3. Section 4 - build + verify `main.py`
4. Section 5 - holdout gate
5. Section 6 - `submission.tar.gz` from working/output directory
